# Paper PoRT Recreated Raw Inverted Route Full Run

This notebook runs the deployable raw inverted-route recreated PoRT path on the full selected WMDP matrix: `original`, `noise_prefix`, and `composite` across `bio`, `chem`, and `cyber`.

It follows notebook 29, which passed the same route at `32` rows/job. The route keeps the initial answer when `postjudge label == 1` and runs rethink otherwise; confidence is recorded but not used for routing.

This is not a smoke test and not an official paper checkpoint metric. It is the full recreated-artifact run for the best deployable route found so far.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Defaults:

- `PORT_MAX_SAMPLES=-1` for full selected datasets
- `PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT=true`
- `PORT_BOOTSTRAP_RECREATED_IF_MISSING=false`
- `PORT_BEST_CLASSIFIER_FEATURE_SET=answer_only`
- route policy: keep initial iff `postjudge label == 1`; otherwise run rethink

Expected row count for the default matrix is `11004` rows: `3668` rows per variant for `original + noise_prefix + composite`.


In [ ]:
os.environ.setdefault('PORT_ARTIFACT_MODE', 'recreated')
os.environ.setdefault('PORT_RUN_NAME', 'paper_port_wmdp_recreated_raw_inverted_route_full_run_phi-1_5')
os.environ.setdefault('PORT_MAX_SAMPLES', '-1')
os.environ.setdefault('PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT', 'true')
os.environ.setdefault('PORT_RECREATED_ARTIFACT_MANIFEST_URL', 'https://raw.githubusercontent.com/toanthangO20/PoRT_LLM_Unlearning-Experiment/artifact-recreated-bootstrap-v1/manifest.json')
os.environ.setdefault('PORT_BOOTSTRAP_RECREATED_IF_MISSING', 'false')
os.environ.setdefault('PORT_RESUME_EXISTING', 'true')
os.environ.setdefault('PORT_FAIL_FAST', 'true')
os.environ.setdefault('PORT_BEST_CLASSIFIER_SAMPLES_PER_DOMAIN', '256')
os.environ.setdefault('PORT_BEST_CLASSIFIER_WRONG_ANSWERS_PER_QUESTION', '3')
os.environ.setdefault('PORT_BEST_CLASSIFIER_FEATURE_SET', 'answer_only')
os.environ.setdefault('PORT_BEST_CLASSIFIER_MAX_FEATURES', '50000')

runtime_keys = [
    'PORT_ARTIFACT_MODE',
    'PORT_RUN_NAME',
    'PORT_WMDP_VARIANTS',
    'PORT_WMDP_DOMAINS',
    'PORT_MAX_SAMPLES',
    'PORT_AUTO_DOWNLOAD_RECREATED_ARTIFACT',
    'PORT_RECREATED_ARTIFACT_MANIFEST_URL',
    'PORT_BOOTSTRAP_RECREATED_IF_MISSING',
    'PORT_CLASSIFIER_CONF_THRESHOLD',
    'PORT_BEST_CLASSIFIER_FEATURE_SET',
    'PORT_RESUME_EXISTING',
    'PORT_FAIL_FAST',
    'PORT_RECREATED_ARTIFACT_DIR',
    'PORT_RECREATED_ARTIFACT_ZIP_URL',
    'PORT_RECREATED_ARTIFACT_ZIP_PATH',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


In [ ]:
from pathlib import Path
import gc
import importlib.util
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


if 'PROJECT_ROOT' not in globals() or not has_project_layout(PROJECT_ROOT):
    PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if 'commit_sha' not in globals():
    commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('Cleared CUDA cache before run.')
except Exception as exc:
    print('CUDA cache cleanup skipped:', exc)

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_recreated_raw_inverted_route_scale_run.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_recreated_raw_inverted_route_scale_run', runner_path)
port_recreated_raw_inverted_route_scale_run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_recreated_raw_inverted_route_scale_run)

result = port_recreated_raw_inverted_route_scale_run.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['run_dir'])
for artifact_name in [
    'artifact_audit.json',
    'run_config.json',
    'summary.json',
    'matrix_summary.csv',
    'matrix_summary.json',
    'summary_by_variant_domain.csv',
    'summary_by_variant_domain.json',
    'all_predictions.csv',
    'failed_jobs.json',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')
